# Lecture 16
## The same workflow in Polars
## Week 6 Wednesday
## Miles Chen, PhD

**How do familiar data-wrangling ideas transfer to another library?**

# Transfer the ideas, then learn the syntax

We have already inspected, filtered, cleaned, summarized, joined, and
reshaped tables in pandas. Today we will express a familiar Titanic analysis
in Polars, paying attention to the same rows, keys, and denominators.

Our main tools are `select`, `filter`, `with_columns`, and `group_by().agg()`.
We will finish with a short demonstration of lazy execution.

You do not need to memorize two complete libraries. Learn how to identify
the operation you need and read the corresponding expression.

# Why learn another library?

Polars offers an expression-based interface, parallel execution, and a lazy
query optimizer. These can be useful for composing and running data
transformations. Depending on the task and the data, Polars can potentially offer much greater performance.

Pandas remains useful, including for existing analysis code and tools that
expect pandas objects.

In [1]:
import polars as pl

titanic = pl.read_csv("../data/titanic.csv")
print("Polars version:", pl.__version__)
print("Rows and columns:", titanic.shape)
titanic.head()

Polars version: 1.40.1
Rows and columns: (891, 12)


passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


# Inspect the table

`shape` gives the row and column counts. `schema` gives column names and
data types. `null_count()` counts missing values in each column.

Polars uses `null` for missing values. We should find the same missing ages
and embarkation codes as in pandas.

In [2]:
titanic.schema

Schema([('passengerid', Int64),
        ('survived', Int64),
        ('pclass', Int64),
        ('name', String),
        ('sex', String),
        ('age', Float64),
        ('sibsp', Int64),
        ('parch', Int64),
        ('ticket', String),
        ('fare', Float64),
        ('cabin', String),
        ('embarked', String)])

In [3]:
titanic.null_count()

passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,177,0,0,0,0,687,2


# A familiar set of verbs

| Goal | R / dplyr | pandas | Polars |
| --- | --- | --- | --- |
| Choose columns | `select()` | `df[[...]]` | `df.select(...)` |
| Keep rows | `filter()` | `df.loc[condition]` | `df.filter(...)` |
| Create or replace a column | `mutate()` | `df["new"] = ...` | `df.with_columns(...)` |
| Sort rows | `arrange()` | `sort_values(...)` | `sort(...)` |
| Summarize groups | `group_by()` + `summarize()` | `groupby(...).agg(...)` | `group_by(...).agg(...)` |

Polars does not have pandas' labeled row index. Keep identifiers such as
`passengerid` in ordinary columns.

# Expressions describe a computation

`pl.col("age")` is an **expression** referring to a column. It is a recipe
that Polars evaluates in a context such as `select`, `filter`, or `agg`.

By contrast, `titanic["age"]` retrieves the actual column as a Series.

We can build an expression before choosing the table on which to evaluate
it. `.alias(...)` names its result.

In [4]:
relatives_expr = pl.col("sibsp") + pl.col("parch")

print("Column expression:", type(pl.col("age")))
print("Actual column:", type(titanic["age"]))

Column expression: <class 'polars.expr.expr.Expr'>
Actual column: <class 'polars.series.series.Series'>


# `select`: choose the output columns

Strings can select existing columns directly. Expressions can select and
compute new output columns. Both calls below return a DataFrame.

In [5]:
titanic.select("passengerid", "name", "age", "fare").head()

passengerid,name,age,fare
i64,str,f64,f64
1,"""Braund, Mr. Owen Harris""",22.0,7.25
2,"""Cumings, Mrs. John Bradley (Fl…",38.0,71.2833
3,"""Heikkinen, Miss. Laina""",26.0,7.925
4,"""Futrelle, Mrs. Jacques Heath (…",35.0,53.1
5,"""Allen, Mr. William Henry""",35.0,8.05


In [6]:
titanic.select(
    "passengerid",
    "name",
    relatives_expr.alias("relatives_aboard"),
).head()

passengerid,name,relatives_aboard
i64,str,i64
1,"""Braund, Mr. Owen Harris""",1
2,"""Cumings, Mrs. John Bradley (Fl…",1
3,"""Heikkinen, Miss. Laina""",0
4,"""Futrelle, Mrs. Jacques Heath (…",1
5,"""Allen, Mr. William Henry""",0


# `filter`: keep rows where an expression is true

The syntax for combining conditions will look familiar: use `&` for
elementwise "and," `|` for "or," and parentheses around comparisons.

Select the columns we want to inspect after filtering.

In [7]:
first_class_women = titanic.filter(
    (pl.col("pclass") == 1) & (pl.col("sex") == "female")
)
print("Passengers selected:", first_class_women.height)
first_class_women.select("passengerid", "name", "age", "survived").head()

Passengers selected: 94


passengerid,name,age,survived
i64,str,f64,i64
2,"""Cumings, Mrs. John Bradley (Fl…",38.0,1
4,"""Futrelle, Mrs. Jacques Heath (…",35.0,1
12,"""Bonnell, Miss. Elizabeth""",58.0,1
32,"""Spencer, Mrs. William Augustus…",null,1
53,"""Harper, Mrs. Henry Sleeper (My…",49.0,1


In [8]:
titanic.filter(pl.col("pclass").is_in([1, 2])).select(
    "passengerid", "pclass", "fare"
).head()

passengerid,pclass,fare
i64,i64,f64
2,1,71.2833
4,1,53.1
7,1,51.8625
10,2,30.0708
12,1,26.55


# `with_columns`: retain columns and add computations

`select` returns the columns you request. `with_columns` retains existing
columns and adds or replaces named columns.

These methods return new DataFrames. Assign a result to a name when you
want to use it later; this call does not change `titanic`.

`sibsp + parch` counts recorded siblings/spouses and parents/children aboard.
We call it `relatives_aboard`. It excludes the passenger themself and does
not measure every possible travel companion.

In [9]:
passengers = titanic.with_columns(
    relatives_expr.alias("relatives_aboard")
)
passengers.select("passengerid", "sibsp", "parch", "relatives_aboard").head()

passengerid,sibsp,parch,relatives_aboard
i64,i64,i64,i64
1,1,0,1
2,1,0,1
3,0,0,0
4,1,0,1
5,0,0,0


# A second step can use the newly created column

Expressions in a single `with_columns` call use that call's input table.
To refer to a column just created in the previous step, use another call.

In [10]:
passengers = passengers.with_columns(
    (pl.col("relatives_aboard") == 0).alias("no_recorded_relatives")
)
passengers.select("passengerid", "relatives_aboard", "no_recorded_relatives").head()

passengerid,relatives_aboard,no_recorded_relatives
i64,i64,bool
1,1,false
2,1,false
3,0,true
4,1,false
5,0,true


# `sort`: make the requested order explicit

Sort fares from highest to lowest. If two fares are equal, use ascending
passenger ID to make the displayed order reproducible.

Polars uses `descending`; pandas uses `ascending`.

In [11]:
passengers.select("passengerid", "name", "fare").sort(
    ["fare", "passengerid"], descending=[True, False]
).head(10)

passengerid,name,fare
i64,str,f64
259,"""Ward, Miss. Anna""",512.3292
680,"""Cardeza, Mr. Thomas Drake Mart…",512.3292
738,"""Lesurer, Mr. Gustave J""",512.3292
28,"""Fortune, Mr. Charles Alexander""",263.0
89,"""Fortune, Miss. Mabel Helen""",263.0
342,"""Fortune, Miss. Alice Elizabeth""",263.0
439,"""Fortune, Mr. Mark""",263.0
312,"""Ryerson, Miss. Emily Borie""",262.375
743,"""Ryerson, Miss. Susan Parker ""S…",262.375


# Your turn: translate a passenger-level analysis

Use the `passengers` table to find **third-class survivors with no recorded
relatives aboard**.

- Keep `passengerid`, `name`, `age`, `fare`, and `relatives_aboard`.
- Sort by fare from highest to lowest, breaking ties by ascending ID.
- Report the total number of selected passengers and show the first ten.
- Should a missing age exclude a passenger from this question? Explain.

In [12]:
# Translate the filtering, selection, and sorting steps here.

# One solution

Age is not part of the selection criteria. We retain a selected passenger
even when their age was not recorded.

In [13]:
selected_passengers = (
    passengers.filter(
        (pl.col("pclass") == 3)
        & (pl.col("survived") == 1)
        & pl.col("no_recorded_relatives")
    )
    .select("passengerid", "name", "age", "fare", "relatives_aboard")
    .sort(["fare", "passengerid"], descending=[True, False])
)
print("Selected passengers:", selected_passengers.height)
selected_passengers.head(10)

Selected passengers: 69


passengerid,name,age,fare,relatives_aboard
i64,str,f64,f64,i64
75,"""Bing, Mr. Lee""",32.0,56.4958,0
510,"""Lang, Mr. Fang""",26.0,56.4958,0
644,"""Foo, Mr. Choong""",null,56.4958,0
693,"""Lam, Mr. Ali""",null,56.4958,0
839,"""Chip, Mr. Chang""",32.0,56.4958,0
208,"""Albimona, Mr. Nassef Cassem""",26.0,18.7875,0
80,"""Dowdell, Miss. Elizabeth""",30.0,12.475,0
778,"""Emanuel, Miss. Virginia Ethel""",5.0,12.475,0
678,"""Turja, Miss. Anna Sofia""",18.0,9.8417,0


# Missing values still require a decision

`is_null()` identifies missing values; `is_not_null()` identifies recorded
values. Use these methods rather than comparing a value to `None`.

`filter` retains rows where its predicate is `True`. Both `False` and
`null` predicates are excluded. A comparison involving a missing age does
not establish that the passenger is an adult or a child.

In [14]:
print("Missing ages:", titanic.filter(pl.col("age").is_null()).height)
print("Recorded ages:", titanic.filter(pl.col("age").is_not_null()).height)

titanic.filter(pl.col("age") >= 18).select("passengerid", "age").head()

Missing ages: 177
Recorded ages: 714


passengerid,age
i64,f64
1,22.0
2,38.0
3,26.0
4,35.0
5,35.0


# Group summaries use expressions too

`group_by` defines the groups; `agg` evaluates summary expressions within
each group. `.alias(...)` gives each result a clear name.

- `pl.len()` counts **rows**, including rows with null values.
- `pl.col("age").count()` counts **non-null ages**.
- `pl.col("age").mean()` averages the non-null ages.
- `pl.col("survived").mean()` gives the survival proportion among recorded
  0/1 outcomes.

The missing numeric values in this CSV are nulls. Floating-point `NaN`
values are a different case, shown in an optional example.

In [15]:
group_summary = (
    titanic.group_by(["pclass", "sex"])
    .agg(
        pl.len().alias("n_passengers"),
        pl.col("survived").count().alias("n_survival"),
        pl.col("survived").mean().alias("survival_rate"),
        pl.col("age").count().alias("n_age"),
        pl.col("age").mean().alias("mean_age"),
    )
    .sort(["pclass", "sex"])
)
group_summary

pclass,sex,n_passengers,n_survival,survival_rate,n_age,mean_age
i64,str,u32,u32,f64,u32,f64
1,"""female""",94,94,0.968085,85,34.611765
1,"""male""",122,122,0.368852,101,41.281386
2,"""female""",76,76,0.921053,74,28.722973
2,"""male""",108,108,0.157407,99,30.740707
3,"""female""",144,144,0.5,102,21.75
3,"""male""",347,347,0.135447,253,26.507589


# Read a result before adding more code

There should be six rows, representing the same class/sex groups as in
lecture 15. Sort the result explicitly; grouping alone does not promise
this display order.

For third-class women, compare `n_passengers`, `n_survival`, and `n_age`.
Which denominator belongs with each mean? Would dropping passengers with
missing ages before grouping answer the same survival question?

In [16]:
group_summary.filter((pl.col("pclass") == 3) & (pl.col("sex") == "female"))

pclass,sex,n_passengers,n_survival,survival_rate,n_age,mean_age
i64,str,u32,u32,f64,u32,f64
3,"""female""",144,144,0.5,102,21.75


In [17]:
group_summary.select(
    pl.col("n_passengers").sum().alias("total_passengers"),
    pl.col("n_survival").sum().alias("recorded_survival_outcomes"),
    pl.col("n_age").sum().alias("recorded_ages"),
)

total_passengers,recorded_survival_outcomes,recorded_ages
u32,u32,u32
891,891,714


# Compute a percentage from the summary

`n_passengers - n_age` counts passengers whose age is missing in each group.
Divide by the group's passenger count to get the fraction, then multiply
by 100 for a percentage.

We can keep the survival analysis population intact while reporting how
much age information is missing.

In [18]:
group_summary = group_summary.with_columns(
    (100 * (pl.col("n_passengers") - pl.col("n_age")) / pl.col("n_passengers"))
    .alias("missing_age_pct")
)
group_summary

pclass,sex,n_passengers,n_survival,survival_rate,n_age,mean_age,missing_age_pct
i64,str,u32,u32,f64,u32,f64,f64
1,"""female""",94,94,0.968085,85,34.611765,9.574468
1,"""male""",122,122,0.368852,101,41.281386,17.213115
2,"""female""",76,76,0.921053,74,28.722973,2.631579
2,"""male""",108,108,0.157407,99,30.740707,8.333333
3,"""female""",144,144,0.5,102,21.75,29.166667
3,"""male""",347,347,0.135447,253,26.507589,27.089337


# Missing group labels: a default worth checking

Polars retains a group for null keys. To obtain this behavior in the pandas
examples from lecture 14, we used `dropna=False`.

The port summary should include the two passengers with no recorded port.
Check the total rather than assuming another library's defaults are the same.

In [19]:
port_summary = titanic.group_by("embarked").agg(
    pl.len().alias("n_passengers")
).sort("embarked")
print("Passengers represented:", port_summary["n_passengers"].sum())
port_summary

Passengers represented: 891


embarked,n_passengers
str,u32
null,2
"""C""",168
"""Q""",77
"""S""",644


# join tables

We will join on `pclass`, keep every group on the left,
and require one label per class on the right.

Polars spells this relationship `validate="m:1"`. As in pandas, the check
rejects repeated lookup keys but does not require all left keys to match.
Check for missing labels separately.

In [20]:
class_labels = pl.DataFrame({
    "pclass": [1, 2, 3],
    "class_name": ["First", "Second", "Third"],
})
analysis_table = group_summary.join(
    class_labels, on="pclass", how="left", validate="m:1"
).sort(["pclass", "sex"])

print("Rows before and after:", group_summary.height, analysis_table.height)
print("Missing class names:", analysis_table["class_name"].null_count())
analysis_table

Rows before and after: 6 6
Missing class names: 0


pclass,sex,n_passengers,n_survival,survival_rate,n_age,mean_age,missing_age_pct,class_name
i64,str,u32,u32,f64,u32,f64,f64,str
1,"""female""",94,94,0.968085,85,34.611765,9.574468,"""First"""
1,"""male""",122,122,0.368852,101,41.281386,17.213115,"""First"""
2,"""female""",76,76,0.921053,74,28.722973,2.631579,"""Second"""
2,"""male""",108,108,0.157407,99,30.740707,8.333333,"""Second"""
3,"""female""",144,144,0.5,102,21.75,29.166667,"""Third"""
3,"""male""",347,347,0.135447,253,26.507589,27.089337,"""Third"""


# Your turn: reproduce the analysis-ready table

Create the same `plot_table` as in lecture 15:

- Select `pclass`, `class_name`, `sex`, `n_passengers`, `n_survival`, and
  `survival_rate` from `analysis_table`.
- Sort by class and sex, and check the number of group rows and the total
  number of passengers represented.
- Explain the third-class female survival rate using its denominator.
- Identify one decision in this workflow that would remain the same in
  pandas, Polars, or R.

In [21]:
# Create plot_table, check its counts, and explain a result.

# One solution

Six group rows represent 891 passengers. Third-class women have 72 recorded
survivors among 144 recorded outcomes: a rate of 0.50.

The library changes how we express the operations. The population,
grouping variables, key relationship, and interpretation remain the same.

In [22]:
plot_table = analysis_table.select(
    "pclass", "class_name", "sex", "n_passengers", "n_survival", "survival_rate"
).sort(["pclass", "sex"])

print("Group rows:", plot_table.height)
print("Distinct class/sex keys:", plot_table.select("pclass", "sex").unique().height)
print("Passengers represented:", plot_table["n_passengers"].sum())
plot_table

Group rows: 6
Distinct class/sex keys: 6
Passengers represented: 891


pclass,class_name,sex,n_passengers,n_survival,survival_rate
i64,str,str,u32,u32,f64
1,"""First""","""female""",94,94,0.968085
1,"""First""","""male""",122,122,0.368852
2,"""Second""","""female""",76,76,0.921053
2,"""Second""","""male""",108,108,0.157407
3,"""Third""","""female""",144,144,0.5
3,"""Third""","""male""",347,347,0.135447


# A brief look at lazy execution

Our DataFrame operations so far have used **eager execution**: each call
evaluates its transformation and returns a result.

With a **LazyFrame**, we build a plan for the transformations. Polars can
optimize the combined plan before computing the result with `collect()`.
`scan_csv` starts a lazy query from a CSV file. Planning can inspect file
metadata or infer a schema; it does not materialize the full result table.

We will compare first- and second-class survival using the same filter and
summaries in both approaches. We do not filter on age.

In [23]:
eager_result = (
    titanic.filter(pl.col("pclass").is_in([1, 2]))
    .group_by(["pclass", "sex"])
    .agg(
        pl.len().alias("n_passengers"),
        pl.col("survived").count().alias("n_survival"),
        pl.col("survived").mean().alias("survival_rate"),
    )
    .sort(["pclass", "sex"])
)
eager_result

pclass,sex,n_passengers,n_survival,survival_rate
i64,str,u32,u32,f64
1,"""female""",94,94,0.968085
1,"""male""",122,122,0.368852
2,"""female""",76,76,0.921053
2,"""male""",108,108,0.157407


In [24]:
lazy_query = (
    pl.scan_csv("../data/titanic.csv")
    .filter(pl.col("pclass").is_in([1, 2]))
    .group_by(["pclass", "sex"])
    .agg(
        pl.len().alias("n_passengers"),
        pl.col("survived").count().alias("n_survival"),
        pl.col("survived").mean().alias("survival_rate"),
    )
    .sort(["pclass", "sex"])
)
type(lazy_query)

polars.lazyframe.frame.LazyFrame

# Inspect the plan, then collect the result

`explain(optimized=True)` describes the optimized plan. Look for a column
projection and the class filter near the CSV scan. Polars can avoid
materializing unused columns and pass the filter into the scan; a CSV scan
may still need to read through the file.

The exact printed plan can vary by version. Focus on the operations, not
memorizing the plan's formatting.

In [25]:
print(lazy_query.explain(optimized=True))

SORT BY [col("pclass"), col("sex")]
  AGGREGATE[maintain_order: false]
    [len().alias("n_passengers"), col("survived").count().alias("n_survival"), col("survived").mean().alias("survival_rate")] BY [col("pclass"), col("sex")]
    FROM
    Csv SCAN [../data/titanic.csv]
    PROJECT 3/12 COLUMNS
    SELECTION: col("pclass").is_in([[1, 2]])
    ESTIMATED ROWS: 973

In [26]:
lazy_result = lazy_query.collect()
lazy_result

pclass,sex,n_passengers,n_survival,survival_rate
i64,str,u32,u32,f64
1,"""female""",94,94,0.968085
1,"""male""",122,122,0.368852
2,"""female""",76,76,0.921053
2,"""male""",108,108,0.157407


In [27]:
from polars.testing import assert_frame_equal

assert_frame_equal(eager_result, lazy_result)
print("Eager and lazy results agree.")
print("Passengers represented:", lazy_result["n_passengers"].sum())

Eager and lazy results agree.
Passengers represented: 400


# What changed, and what did not?

The lazy query organizes how the computation is executed. It does not
change the intended filter, group definitions, or denominators: both
versions describe the same 400 first- and second-class passengers.

Starting with `scan_csv` gives the optimizer access to the file scan.
Calling `.lazy()` on a DataFrame is also possible, but that DataFrame has
already been loaded into memory.

We have not timed these examples. Small-data timings are not evidence that
one library will always be faster for every workflow.

# From tables to visualization

We now have the same analysis-ready structure in two libraries: one row per
class/sex group, explicit labels, a survival rate, and its denominator.

In the visualization lectures, we can map class to an axis, survival rate
to height, and sex to color. The choice of chart comes after deciding what
the rows and summaries mean. Use the table or array input supported by the
particular plotting function.

**Exit check:** What is an expression? How does `select` differ from
`with_columns`? Why are `len` and `count` sometimes different? What does
`collect()` do?

# reshaping the rates with `pivot()`

In lecture 15, `pivot` arranged one rate per class/sex group into a wide
table. Polars uses `on` for the column whose values become column labels.

We already have one value per class/sex key, so no aggregation is needed.
Keep the fuller `analysis_table` to retain denominators.

In [28]:
survival_wide = group_summary.pivot(
    on="sex", index="pclass", values="survival_rate"
).sort("pclass")
survival_wide

pclass,female,male
i64,f64,f64
1,0.968085,0.368852
2,0.921053,0.157407
3,0.5,0.135447


# `unpivot` is the wide-to-long operation

Here `on` names the measurement columns to gather, and `index` names the
identifier columns to keep. These are ordinary columns; Polars has no
pandas-style row index to reset.

The `sex` values come from column labels. The numeric `pclass` values remain
in their original identifier column.

In [29]:
survival_long = survival_wide.unpivot(
    on=["female", "male"],
    index="pclass",
    variable_name="sex",
    value_name="survival_rate",
).sort(["pclass", "sex"])
survival_long

pclass,sex,survival_rate
i64,str,f64
1,"""female""",0.968085
1,"""male""",0.368852
2,"""female""",0.921053
2,"""male""",0.157407
3,"""female""",0.5
3,"""male""",0.135447


# conditions must account for unknown values

Suppose we want an age label using 18 as the threshold. Handle missing ages
first. Otherwise, an `otherwise(pl.lit("child"))` branch would incorrectly classify
a missing age as evidence that a passenger was a child.

In expression contexts such as `then`, a bare string is interpreted as a
column name. Use `pl.lit("adult")` to supply a literal string. Numeric
literals can be used directly in arithmetic and comparisons, as above.

In [30]:
age_labeled = titanic.with_columns(
    pl.when(pl.col("age").is_null())
    .then(pl.lit("not recorded"))
    .when(pl.col("age") >= 18)
    .then(pl.lit("adult"))
    .otherwise(pl.lit("child"))
    .alias("age_status")
)
age_labeled.group_by("age_status").agg(
    pl.len().alias("n_passengers")
).sort("age_status")

age_status,n_passengers
str,u32
"""adult""",601
"""child""",113
"""not recorded""",177


# `null` and `NaN` are different

A null is a missing value. `NaN` is a special floating-point value that can
arise from numeric operations. Polars does not treat `NaN` as null:
`count` includes it, and a mean can be `NaN` even after nulls are skipped.

Inspect the values and their source before choosing `fill_null`, `fill_nan`,
or a filtering rule. They address different conditions.

In [31]:
numeric_example = pl.DataFrame({"x": [1.0, None, float("nan")]})
numeric_example.select(
    pl.len().alias("n_rows"),
    pl.col("x").count().alias("n_non_null"),
    pl.col("x").null_count().alias("n_null"),
    pl.col("x").is_nan().sum().alias("n_nan"),
    pl.col("x").mean().alias("mean_x"),
)

n_rows,n_non_null,n_null,n_nan,mean_x
u32,u32,u32,u32,f64
3,2,1,1,NaN


# join defaults can differ across libraries

Polars does not match null join keys by default. Pandas can match missing
keys to missing keys. When a match between missing keys has an appropriate
meaning, Polars provides `nulls_equal=True`.

Our class-label join uses complete keys, so this difference does not affect
the result. Missing keys, repeated keys, and unmatched keys are still
separate things to check.

# Optional: streaming execution

A lazy query can request `collect(engine="streaming")`. Supported parts of
the query can process data in batches, which can reduce working memory.
Support depends on the operations and version, and some work may fall back
to the in-memory engine.

`collect` still returns a materialized DataFrame. Streaming is not a promise
that an arbitrarily large final result will fit in memory. This small
example illustrates the API, not a memory or speed benefit.

In [32]:
streaming_result = lazy_query.collect(engine="streaming")
assert_frame_equal(eager_result, streaming_result)
streaming_result

pclass,sex,n_passengers,n_survival,survival_rate
i64,str,u32,u32,f64
1,"""female""",94,94,0.968085
1,"""male""",122,122,0.368852
2,"""female""",76,76,0.921053
2,"""male""",108,108,0.157407


# References

- [Polars: expressions and contexts](https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/)
- [Polars: aggregation](https://docs.pola.rs/user-guide/expressions/aggregation/)
- [Polars: missing data](https://docs.pola.rs/user-guide/expressions/missing-data/)
- [Polars: joins](https://docs.pola.rs/user-guide/transformations/joins/)
- [Polars: lazy execution](https://docs.pola.rs/user-guide/concepts/lazy-api/)
- [Polars: streaming](https://docs.pola.rs/user-guide/concepts/streaming/)